# Curator Verification (Stage 0.5)

In this notebook we test `agent/segment.py` (raw transcript → schema) and
`agent/curate.py` (LLM-based clip selection) against real transcript content,
via a live Anthropic API call.

We deliberately separate this notebook from `01_development.ipynb` and `02_verification.ipynb`because the curator only needs `anthropic`, not torch/whisperx/ffmpeg. No need to use GPU for this.

**Tested against:** a real NASA subject-matter-expert video transcript in
broadcast timecode format (`HH:MM:SS:FF` ranges) — a third real transcript
shape, distinct from the speaker-labeled and YouTube-caption formats tested
elsewhere.

**Real issues hit and fixed along the way:** an API key without a workspace
scope required an `anthropic-workspace-id` header on every request (fixed by
recreating the key scoped to one workspace); extended thinking put a
`ThinkingBlock` before the `TextBlock` in the response, breaking a
`response.content[0].text` assumption (fixed by finding the text block by
type instead of position).

**Result:** on a 21-segment excerpt with genuine topic shifts, the curator
correctly identified 5 clip-worthy spans with accurate boundaries — including
correctly keeping together a setup-and-payoff passage an earlier, narrower
test had incorrectly split in two. One minor issue remained: a single
transition sentence landed at the end of one clip instead of the start of
the next — the scale of correction a human review step exists to catch.

**Setup:** `pip install anthropic`, then an `ANTHROPIC_API_KEY` secret
(Colab: 🔑 icon, scoped to a single workspace).

In [1]:
!rm -rf video-clipping-agent

In [2]:
!git clone https://github.com/AIanumel2025/video-clipping-agent.git
%cd video-clipping-agent

Cloning into 'video-clipping-agent'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 101 (delta 28), reused 86 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 35.98 MiB | 25.37 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/video-clipping-agent


In [3]:
!pip install anthropic -q

In [4]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [5]:
from agent.segment import segment_raw_transcript

raw = """Dr. Lesley Ott SME Transcript

Narration: Dr. Lesley Ott

Transcript:

00:00:44:18 - 00:00:45:22

The Earth is

00:00:45:22 - 00:00:46:07

the most

00:00:46:07 - 00:00:47:16

important planet that we study,

00:00:47:16 - 00:00:49:21

at least in my opinion.

00:00:49:21 - 00:00:52:20

And the work that we do from space

00:00:52:20 - 00:00:54:03

is incredibly valuable

00:00:54:03 - 00:00:55:23

in giving that this global vantage point,

00:00:55:23 - 00:00:57:10

the ability to see everything,

00:00:57:10 - 00:00:58:20

It's like going to the world,

00:00:58:20 - 00:01:00:10

you know, the top of a big hill

00:01:00:10 - 00:01:01:05

and you get to look down

00:01:01:05 - 00:01:01:21

and see everything,

00:01:01:21 - 00:01:03:08

but there's no place we can do that.

00:01:03:08 - 00:01:04:09

What we're actually

00:01:04:09 - 00:01:05:20

on the surface of the earth.

00:01:05:20 - 00:01:07:15

And so the work that NASA does

00:01:07:15 - 00:01:08:21

in exploring space

00:01:08:21 - 00:01:10:01

and finding new ways

00:01:10:01 - 00:01:10:21

to get instruments

00:01:10:21 - 00:01:12:04

that look back at Earth

00:01:12:04 - 00:01:13:00

have given us this

00:01:13:00 - 00:01:14:13

incredibly rich dataset

00:01:14:13 - 00:01:15:20

that help us understand

00:01:15:20 - 00:01:16:08

the world

00:01:16:08 - 00:01:17:16

and how it's changing over

00:01:17:16 - 00:01:20:08

the course of multiple decades now

00:01:27:07 - 00:01:28:16

you know, when NASA's first started

00:01:28:16 - 00:01:30:19

sending people up into space,

00:01:30:19 - 00:01:31:22

they would look down at the Earth

00:01:31:22 - 00:01:32:19

and they would see things

00:01:32:19 - 00:01:34:05

that we see from the ground.

00:01:34:05 - 00:01:35:03

But this whole different

00:01:35:03 - 00:01:35:20

perspective, right?

00:01:35:20 - 00:01:36:23

They could see the movement

00:01:36:23 - 00:01:38:22

of these big clouds and weather systems.

00:01:38:22 - 00:01:41:03

They could see lights at night. Right.

00:01:41:03 - 00:01:42:10

And so over time,

00:01:42:10 - 00:01:43:09

that guided

00:01:43:09 - 00:01:45:02

the development of instruments

00:01:45:02 - 00:01:46:08

that we've sent into space

00:01:46:08 - 00:01:47:17

that look back at our planet,

00:01:47:17 - 00:01:48:21

that are able to monitor

00:01:48:21 - 00:01:49:16

weather systems,

00:01:49:16 - 00:01:51:22

that are able to monitor human activity.

00:01:51:22 - 00:01:54:09

So now those data sets have provided

00:01:54:09 - 00:01:56:04

this foundation of understand

00:01:56:04 - 00:01:58:01

how the Earth functions as a planet

00:01:58:01 - 00:01:58:20

over the course of the

00:01:58:20 - 00:02:01:00

past several decades.

00:02:07:19 - 00:02:08:14

Studying the Earth as a

00:02:08:14 - 00:02:10:03

system is incredibly important

00:02:10:03 - 00:02:11:04

because the more we look

00:02:11:04 - 00:02:12:22

at individual variables

00:02:12:22 - 00:02:14:16

like rainfall or air

00:02:14:16 - 00:02:16:02

pollutants or temperature,

00:02:16:02 - 00:02:17:11

we realize how interconnected

00:02:17:11 - 00:02:20:11

the Earth is as a planet, as a system,

00:02:20:11 - 00:02:21:22

right when we warm the planet,

00:02:21:22 - 00:02:23:05

when we increase temperatures,

00:02:23:05 - 00:02:24:22

we don't just get warmth

00:02:24:22 - 00:02:26:13

and balmy days, right?

00:02:26:13 - 00:02:28:08

We get changes in weather systems.

00:02:28:08 - 00:02:29:15

We get changes

00:02:29:15 - 00:02:32:08

in the acuteness of pollution episodes

00:02:32:08 - 00:02:34:02

that send people to the hospital.

00:02:34:02 - 00:02:35:07

We change the frequency

00:02:35:07 - 00:02:37:01

and the patterns of wildfires

00:02:37:01 - 00:02:38:04

so understanding

00:02:38:04 - 00:02:39:19

all those different variables

00:02:39:19 - 00:02:41:06

and how connected they are,

00:02:41:06 - 00:02:42:19

that's one of the most special things

00:02:42:19 - 00:02:43:08

that we get

00:02:43:08 - 00:02:45:00

from this record of Earth observations

00:02:45:00 - 00:02:46:07

and this really impressive

00:02:46:07 - 00:02:47:19

global perspective

00:02:47:19 - 00:02:49:09

that NASA's satellites give us

00:03:33:11 - 00:03:34:16

So atmospheric chemistry is a

00:03:34:16 - 00:03:36:02

lot like the chemistry

00:03:36:02 - 00:03:37:01

that you probably learned

00:03:37:01 - 00:03:37:20

about in high school.

00:03:37:20 - 00:03:38:10

But when we talk about

00:03:38:10 - 00:03:39:13

atmospheric chemistry,

00:03:39:13 - 00:03:40:20

we're talking about the way

00:03:40:20 - 00:03:43:16

that that molecules in the form of gases

00:03:43:16 - 00:03:44:14

react with each other

00:03:44:14 - 00:03:46:12

to form new molecules,

00:03:46:12 - 00:03:46:22

many of them

00:03:46:22 - 00:03:48:15

in very small concentrations.

00:03:48:15 - 00:03:50:08

But those can be incredibly important

00:03:50:08 - 00:03:52:04

and affecting things like air quality

00:03:52:04 - 00:03:53:18

pollution and public health

00:03:53:18 - 00:03:56:00

and the evolution of climate change

00:04:43:09 - 00:04:44:16

So greenhouse gases are this

00:04:44:16 - 00:04:46:09

really important type of gas

00:04:46:09 - 00:04:47:06

that's present in the atmosphere

00:04:47:06 - 00:04:49:06

in really, really small concentrations.

00:04:49:06 - 00:04:49:22

But it's important

00:04:49:22 - 00:04:50:18

because they're very,

00:04:50:18 - 00:04:51:12

very efficient

00:04:51:12 - 00:04:52:13

at trapping

00:04:52:13 - 00:04:53:13

long wave heat

00:04:53:13 - 00:04:56:00

that's emitted from the Earth's surface.

00:05:16:03 - 00:05:17:06

So some of the most important

00:05:17:06 - 00:05:18:05

greenhouse gases

00:05:18:05 - 00:05:18:23

that are influenced by

00:05:18:23 - 00:05:20:03

human activities are carbon

00:05:20:03 - 00:05:21:10

dioxide or CO2,

00:05:21:10 - 00:05:23:11

as you may have heard of, called methane,

00:05:23:11 - 00:05:26:03

nitrous oxide or N2O

00:05:26:03 - 00:05:27:05

and also water vapor.

00:05:55:10 - 00:05:56:23

So methane is really powerful.

00:05:56:23 - 00:05:57:14

Greenhouse gas

00:05:57:14 - 00:05:58:21

is actually more powerful

00:05:58:21 - 00:06:00:12

on a molecule by molecule

00:06:00:12 - 00:06:01:23

basis than carbon dioxide,

00:07:01:17 - 00:07:02:22

So another reason that methane

00:07:02:22 - 00:07:04:08

is really important right now

00:07:04:08 - 00:07:04:20

is that

00:07:04:20 - 00:07:07:13

policymakers are targeting

00:07:07:13 - 00:07:09:00

methane emissions as a

00:07:09:00 - 00:07:11:05

as a major effort to reduce

00:07:11:05 - 00:07:13:00

emissions from methane in the near term.

00:07:36:06 - 00:07:37:20

that'll make life a little bit safer

00:07:37:20 - 00:07:38:14

and help us reduce

00:07:38:14 - 00:07:41:00

climate change at the same time.
"""

segmented = segment_raw_transcript(raw, video_id="nasa_curator_test_v2")
print(f"{len(segmented['segments'])} segments ready for curation")

Segmented into 21 segments (timecode-range lines (SRT/broadcast-style) -- timecodes and any header text before the first one discarded)
21 segments ready for curation


In [6]:
from agent.curate import curate_transcript

result = curate_transcript(segmented)

for s in result["segments"]:
    if s["make_clip"]:
        print(f"{s['id']}: \"{s['clip_title']}\"")
        print(f"  {s['text']}")
        print()

Curator flagged 5 clip(s) from 21 segments
seg_001: "Why Earth is the most important planet to study"
  The Earth is the most important planet that we study, at least in my opinion. And the work that we do from space is incredibly valuable in giving that this global vantage point, the ability to see everything, It's like going to the world, you know, the top of a big hill and you get to look down and see everything, but there's no place we can do that. What we're actually on the surface of the earth.

seg_004: "How NASA's view from space transformed Earth science"
  And so the work that NASA does in exploring space and finding new ways to get instruments that look back at Earth have given us this incredibly rich dataset that help us understand the world and how it's changing over the course of multiple decades now you know, when NASA's first started sending people up into space, they would look down at the Earth and they would see things that we see from the ground. But this whole diff